# RAG Query — Hybrid + HyDE + Parent-expand + Rerank + LLM
Pipeline: `query` → normalize → hybrid search (nhanh goc) + HyDE paraphrase → hybrid search lan 2
→ merge RRF → parent aggregation → top-K parents → expand children → CrossEncoder rerank → top-N → LLM.
Yeu cau: da chay `data_pipeline.ipynb` (co `output/` + Qdrant `rag_docs`).
Sua bien `query` o cell Config roi chay tu tren xuong.

In [ ]:
# !pip install langchain-qdrant langchain-ollama fastembed sentence-transformers qdrant-client = boto3

In [95]:
!pip install boto3

   ---------------------------------------- 0.0/16.0 MB ? eta -:--:--
   ----------- ---------------------------- 4.7/16.0 MB 28.6 MB/s eta 0:00:01
   ------------------------ --------------- 9.7/16.0 MB 43.1 MB/s eta 0:00:01
   ------------------------ --------------- 10.0/16.0 MB 20.7 MB/s eta 0:00:01
   --------------------------- ------------ 11.0/16.0 MB 14.3 MB/s eta 0:00:01
   -------------------------------------- - 15.5/16.0 MB 16.0 MB/s eta 0:00:01
   ---------------------------------------- 16.0/16.0 MB 14.8 MB/s  0:00:01

  Attempting uninstall: botocore

    Found existing installation: botocore 1.34.69

   ---------------------------------------- 0/3 [botocore]
   ---------------------------------------- 0/3 [botocore]
   ---------------------------------------- 0/3 [botocore]
   ---------------------------------------- 0/3 [botocore]
    Uninstalling botocore-1.34.69:
   ---------------------------------------- 0/3 [botocore]
   ---------------------------------------- 0

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
aiobotocore 2.12.3 requires botocore<1.34.70,>=1.34.41, but you have botocore 1.43.106 which is incompatible.


{'ResponseMetadata': {'RequestId': 'TAFDEDYA5SAQPHF2', 'HostId': '6LALfhXk5PwY/SzDuCraxqDGgmpIiicRAUibGF5sULIPCaO3cvlKnJPlkkH9AepT1AuA917IHgqNcTF7c9roaA/yAZj4jlUe', 'HTTPStatusCode': 200, 'HTTPHeaders': {'x-amz-id-2': '6LALfhXk5PwY/SzDuCraxqDGgmpIiicRAUibGF5sULIPCaO3cvlKnJPlkkH9AepT1AuA917IHgqNcTF7c9roaA/yAZj4jlUe', 'x-amz-request-id': 'TAFDEDYA5SAQPHF2', 'server': 'AmazonS3', 'date': 'Thu, 01 Oct 2026 15:34:20 GMT', 'x-amz-bucket-region': 'us-east-1', 'content-type': 'application/xml', 'transfer-encoding': 'chunked'}, 'RetryAttempts': 1}, 'IsTruncated': False, 'Name': 'amzn-s3-hoangdep', 'Prefix': '', 'MaxKeys': 1000, 'EncodingType': 'url', 'KeyCount': 0}


In [74]:
# ============================================================
# 0. CONFIG (sua query cua ban o day)
# ============================================================
QDRANT_URL = "http://localhost:6333"
COLLECTION = "rag_docs"
EMBED_MODEL = "RizwanMalik/qwen3-vl-embedding-2b:q4_0-q8_0"
LLM_MODEL = "qwen3:1.7b"
RERANKER_MODEL = "Qwen/Qwen3-Reranker-0.6B"

query = "Compare for me nikon 24-70 f/4 and 24-200 f/4 -f6.3 nikon z  "

# --- Pipeline params ---
NORMALIZE_WS = True       # chuan hoa whitespace (strip + collapse), giu nguyen case
N_PARAPHRASES = 3         # so cau paraphrase HyDE sinh ra
HYDE_K = 10               # top-k moi nhanh hybrid search
FUSED_K = 20              # top-k sau fusion RRF
RRF_K = 60                # hang so RRF
TOP_PARENTS = 3           # so parent giu lai
MAX_CHILD_PER_PARENT = 10 # tran children moi parent khi expand
TOP_N = 5                 # so context cuoi dua vao LLM


In [26]:
OLLAMA_URL_CHAT = "http://localhost:11434/api/chat"
model = LLM_MODEL  # lấy từ cell Config


In [27]:
from langchain_ollama import ChatOllama

llm = ChatOllama(
    model=model,
    temperature=0,
)

In [28]:
from langchain_ollama import OllamaEmbeddings
EMBED_MODEL = "RizwanMalik/qwen3-vl-embedding-2b:q4_0-q8_0"
embeddings = OllamaEmbeddings(
    model=EMBED_MODEL
)

In [29]:
from langchain_qdrant import QdrantVectorStore
from qdrant_client import QdrantClient
from qdrant_client.http.models import Distance, VectorParams
from langchain_qdrant import (
    QdrantVectorStore,
    RetrievalMode,
)
from langchain_qdrant import FastEmbedSparse
sparse_embeddings = FastEmbedSparse(
    model_name="Qdrant/bm25"
)
QDRANT_URL = "http://localhost:6333"
client = QdrantClient(QDRANT_URL)



vector_store = QdrantVectorStore(
    client=client,
    collection_name=COLLECTION,

    # Dense
    embedding=embeddings,

    # Sparse
    sparse_embedding=sparse_embeddings,

    # Hybrid
    retrieval_mode=RetrievalMode.HYBRID,

    # Tên 2 vector trong Qdrant
    vector_name="dense",
    sparse_vector_name="sparse",
    content_payload_key="page_content",
    metadata_payload_key="metadata",
)

In [30]:
info = client.get_collection(COLLECTION)

print(info)
print(info.config.params.vectors)
print(info.config.params.sparse_vectors)

status=<CollectionStatus.GREEN: 'green'> optimizer_status=<OptimizersStatusOneOf.OK: 'ok'> warnings=None indexed_vectors_count=123 points_count=123 segments_count=6 config=CollectionConfig(params=CollectionParams(vectors={'dense': VectorParams(size=2048, distance=<Distance.COSINE: 'Cosine'>, hnsw_config=None, quantization_config=None, on_disk=None, memory=None, datatype=None, multivector_config=None)}, shard_number=1, sharding_method=None, replication_factor=1, write_consistency_factor=1, read_fan_out_factor=None, read_fan_out_delay_ms=None, on_disk_payload=True, payload=None, sparse_vectors={'sparse': SparseVectorParams(index=None, modifier=None)}), hnsw_config=HnswConfig(m=16, ef_construct=100, full_scan_threshold=10000, max_indexing_threads=0, on_disk=False, memory=None, payload_m=None, inline_storage=None), optimizer_config=OptimizersConfig(deleted_threshold=0.2, vacuum_min_vector_number=1000, default_segment_number=0, max_segment_size=None, memmap_threshold=None, indexing_threshol

In [31]:
print("retrieval_mode =", vector_store.retrieval_mode)
print("vector_name =", repr(vector_store.vector_name))
print("sparse_vector_name =", repr(vector_store.sparse_vector_name))

retrieval_mode = RetrievalMode.HYBRID
vector_name = 'dense'
sparse_vector_name = 'sparse'


In [75]:
# ============================================================
# 1. QUERY NORMALIZATION (whitespace-only: strip + collapse)
# ============================================================
import re

if NORMALIZE_WS:
    query_norm = re.sub(r"\s+", " ", query).strip()
else:
    query_norm = query

print(f"query      = {query!r}")
print(f"query_norm = {query_norm!r}")


query      = 'Compare for me nikon 24-70 f/4 and 24-200 f/4 -f6.3 nikon z  '
query_norm = 'Compare for me nikon 24-70 f/4 and 24-200 f/4 -f6.3 nikon z'


In [76]:
# ============================================================
# 2. HYBRID SEARCH — nhanh goc (Dense + Sparse + DBSF)
# ============================================================
from datetime import datetime, timezone
from qdrant_client import models

now = datetime.now(timezone.utc)

# Vi du filter (khong bat buoc) — bo comment va sua value neu can:
# qdrant_filter = models.Filter(
#     must=[
#         models.FieldCondition(
#             key="metadata.source_file",
#             match=models.MatchValue(value="ten_file.pdf"),
#         ),
#     ]
# )
qdrant_filter = None

res_base = vector_store.similarity_search_with_relevance_scores(
    query=query_norm,  # da chuan hoa tu cell 1
    k=HYDE_K,
    filter=qdrant_filter,
    hybrid_fusion=models.FusionQuery(fusion=models.Fusion.DBSF),
)

print(f"base results: {len(res_base)}")
for doc, score in res_base[:5]:
    print(f"{score:.4f}", doc.metadata["chunk_id"], doc.metadata.get("parent_title"))


base results: 10
1.2346 chunk_0117 Specifications
1.2124 chunk_0114 Specifications
1.1441 chunk_0120 Specifications
1.0932 chunk_0012 ❚❚ Nikon User Support
1.0324 chunk_0011 ❚❚ Nikon User Support


C:\Users\Admin\AppData\Local\Temp\ipykernel_32504\381193172.py:20: UserWarning: Relevance scores must be between 0 and 1, got [(Document(metadata={'type': 'chunk', 'chunk_id': 'chunk_0117', 'chunk_index': 117, 'parent_id': 'parent_2bdaae1b71e4', 'parent_title': 'Specifications', 'section_path': 'Specifications', 'source_file': 'Z5UM_EU(En)04.pdf', 'num_tokens': 477, 'page_nos': [72, 73], 'images': [], 'tables': [{'table_id': 'table_0026', 'csv': 'tables/table_0026.csv', 'html': 'tables/table_0026.html', 'page_no': 72, 'doc_item_ref': '#/tables/25'}, {'table_id': 'table_0027', 'csv': 'tables/table_0027.csv', 'html': 'tables/table_0027.html', 'page_no': 73, 'doc_item_ref': '#/tables/26'}], 'created_at': '2026-10-01T00:46:58+07:00', 'updated_at': None, 'file': 'chunks/chunk_0117.md', '_id': 117, '_collection_name': 'rag_docs'}, page_content="Specifications\nMount, 1 = Nikon Z mount. Focal length, 1 = 24-200mm. Maximumaperture, 1 = f/4 - 6.3. Lens construction, 1 = 19 elements in 15 groups

In [77]:
# ============================================================
# 3. HyDE — sinh paraphrase tu query cua user (bang llm san co)
# ============================================================
hyde_prompt = (
    "Rewrite the following question into "
    f"{N_PARAPHRASES} diverse search queries. "
    "Rules: one query per line, no numbering, no explanation, "
    "keep technical terms and model names unchanged.\n\n"
    f"Question: {query_norm}"
)
hyde_raw = llm.invoke(hyde_prompt).content
print(hyde_raw)

paraphrases = []
for line in hyde_raw.splitlines():
    line = line.strip().lstrip("0123456789.-) ").strip()
    if line and line.lower() != query_norm.lower():
        paraphrases.append(line)

# dedup giu thu tu + gioi han N
seen = set()
queries_hyde = []
for q in paraphrases:
    if q not in seen:
        seen.add(q)
        queries_hyde.append(q)
    if len(queries_hyde) >= N_PARAPHRASES:
        break

print(f"\nparaphrases kept: {len(queries_hyde)}")
for q in queries_hyde:
    print("-", q)


nikon 24-70 f/4 vs 24-200 f/4 -f/6.3 nikon z focal length and aperture comparison  
nikon z 24-70 f/4 vs 24-200 f/4 -f/6.3 nikon z zoom range and f-stop performance  
nikon z 24-70 f/4 vs 24-200 f/4 -f/6.3 nikon z sensor size and autofocus capabilities

paraphrases kept: 3
- nikon 24-70 f/4 vs 24-200 f/4 -f/6.3 nikon z focal length and aperture comparison
- nikon z 24-70 f/4 vs 24-200 f/4 -f/6.3 nikon z zoom range and f-stop performance
- nikon z 24-70 f/4 vs 24-200 f/4 -f/6.3 nikon z sensor size and autofocus capabilities


In [78]:
# ============================================================
# 4. HYBRID SEARCH lan 2 — moi paraphrase mot nhanh
# ============================================================
res_hyde = {}  # paraphrase -> list[(doc, score)]

for q in queries_hyde:
    res = vector_store.similarity_search_with_relevance_scores(
        query=q,
        k=HYDE_K,
        filter=qdrant_filter,
        hybrid_fusion=models.FusionQuery(fusion=models.Fusion.DBSF),
    )
    res_hyde[q] = res
    print(f"{len(res)} hits :: {q!r}")


C:\Users\Admin\AppData\Local\Temp\ipykernel_32504\2825460689.py:7: UserWarning: Relevance scores must be between 0 and 1, got [(Document(metadata={'type': 'chunk', 'chunk_id': 'chunk_0117', 'chunk_index': 117, 'parent_id': 'parent_2bdaae1b71e4', 'parent_title': 'Specifications', 'section_path': 'Specifications', 'source_file': 'Z5UM_EU(En)04.pdf', 'num_tokens': 477, 'page_nos': [72, 73], 'images': [], 'tables': [{'table_id': 'table_0026', 'csv': 'tables/table_0026.csv', 'html': 'tables/table_0026.html', 'page_no': 72, 'doc_item_ref': '#/tables/25'}, {'table_id': 'table_0027', 'csv': 'tables/table_0027.csv', 'html': 'tables/table_0027.html', 'page_no': 73, 'doc_item_ref': '#/tables/26'}], 'created_at': '2026-10-01T00:46:58+07:00', 'updated_at': None, 'file': 'chunks/chunk_0117.md', '_id': 117, '_collection_name': 'rag_docs'}, page_content="Specifications\nMount, 1 = Nikon Z mount. Focal length, 1 = 24-200mm. Maximumaperture, 1 = f/4 - 6.3. Lens construction, 1 = 19 elements in 15 groups

10 hits :: 'nikon 24-70 f/4 vs 24-200 f/4 -f/6.3 nikon z focal length and aperture comparison'
10 hits :: 'nikon z 24-70 f/4 vs 24-200 f/4 -f/6.3 nikon z zoom range and f-stop performance'
10 hits :: 'nikon z 24-70 f/4 vs 24-200 f/4 -f/6.3 nikon z sensor size and autofocus capabilities'


C:\Users\Admin\AppData\Local\Temp\ipykernel_32504\2825460689.py:7: UserWarning: Relevance scores must be between 0 and 1, got [(Document(metadata={'type': 'chunk', 'chunk_id': 'chunk_0120', 'chunk_index': 120, 'parent_id': 'parent_2bdaae1b71e4', 'parent_title': 'Specifications', 'section_path': 'Specifications', 'source_file': 'Z5UM_EU(En)04.pdf', 'num_tokens': 341, 'page_nos': [75], 'images': [], 'tables': [{'table_id': 'table_0028', 'csv': 'tables/table_0028.csv', 'html': 'tables/table_0028.html', 'page_no': 75, 'doc_item_ref': '#/tables/27'}], 'created_at': '2026-10-01T00:46:58+07:00', 'updated_at': None, 'file': 'chunks/chunk_0120.md', '_id': 120, '_collection_name': 'rag_docs'}, page_content="Specifications\nFocal length, Nikon Z mount = 24-50mm. Maximumaperture, Nikon Z mount = f/4 - 6.3. Lens construction, Nikon Z mount = 11 elements in 10 groups (including 2 EDelements and 3 aspherical elements). Angle of view, Nikon Z mount = • FX format: 84°-47° • DX format: 61°-31° 30'. Foca

In [79]:
# ============================================================
# 5. MERGE / DEDUPLICATE — RRF fusion (nhanh goc + cac nhanh HyDE)
# ============================================================
from collections import defaultdict

# moi nhanh la mot ranking list; rank bat dau tu 1
rank_lists = {"__base__": res_base, **res_hyde}

rrf_scores = defaultdict(float)
doc_by_id = {}
branch_hit = defaultdict(set)

for branch, results in rank_lists.items():
    for rank, (doc, _score) in enumerate(results, start=1):
        cid = doc.metadata["chunk_id"]
        rrf_scores[cid] += 1.0 / (RRF_K + rank)
        branch_hit[cid].add(branch)
        # giu doc co score goc cao nhat lam dai dien
        if cid not in doc_by_id:
            doc_by_id[cid] = doc

fused_ans = sorted(
    ((doc_by_id[cid], score) for cid, score in rrf_scores.items()),
    key=lambda x: x[1],
    reverse=True,
)[:FUSED_K]

ans = fused_ans  # downstream (parent group -> rerank) dung bien nay

print(f"fused: {len(fused_ans)} (from {len(rank_lists)} branches)")
for rank, (doc, score) in enumerate(fused_ans, start=1):
    print(
        f"{rank:2d} fused={score:.4f} "
        f"branches={len(branch_hit[doc.metadata['chunk_id']])} "
        f"{doc.metadata['chunk_id']} {doc.metadata.get('parent_title')} "
        f"{doc.metadata.get('page_nos')}"
    )


fused: 15 (from 4 branches)
 1 fused=0.0651 branches=4 chunk_0117 Specifications [72, 73]
 2 fused=0.0643 branches=4 chunk_0120 Specifications [75]
 3 fused=0.0643 branches=4 chunk_0114 Specifications [70]
 4 fused=0.0625 branches=4 chunk_0012 ❚❚ Nikon User Support [7]
 5 fused=0.0613 branches=4 chunk_0011 ❚❚ Nikon User Support [7]
 6 fused=0.0582 branches=4 chunk_0080 Nikon Z 5 Digital Camera []
 7 fused=0.0452 branches=3 chunk_0070 Nikon Z 5 Digital Camera [42, 43]
 8 fused=0.0301 branches=2 chunk_0118 NIKKOR Z 24-50mm f/4-6.3 Lens []
 9 fused=0.0294 branches=2 chunk_0013 ❚❚ Nikon User Support [7, 8]
10 fused=0.0294 branches=2 chunk_0116 Parts of the Lens: Names and Functions [71]
11 fused=0.0292 branches=2 chunk_0119 Parts of the Lens: Names and Functions [74]
12 fused=0.0292 branches=2 chunk_0074 Nikon Z 5 Digital Camera [47]
13 fused=0.0152 branches=1 chunk_0115 NIKKOR Z 24-200mm f/4-6.3 VR Lens []
14 fused=0.0143 branches=1 chunk_0113 Parts of the Lens: Names and Functions [69]
1

In [80]:
# ============================================================
# 6. PARENT AGGREGATION — max / mean / count tren fused_ans
# ============================================================
from collections import defaultdict

parent_groups = defaultdict(list)

for doc, score in ans:
    parent_groups[doc.metadata["parent_id"]].append({"doc": doc, "score": score})

parent_scores = {}
for parent_id, items in parent_groups.items():
    scores = [it["score"] for it in items]
    parent_scores[parent_id] = {
        "title": items[0]["doc"].metadata.get("parent_title"),
        "max_score": max(scores),
        "mean_score": sum(scores) / len(scores),
        "count": len(scores),
    }

for pid, info in sorted(parent_scores.items(), key=lambda kv: kv[1]["max_score"], reverse=True):
    print(pid, info)


parent_2bdaae1b71e4 {'title': 'Specifications', 'max_score': 0.06505334374186833, 'mean_score': 0.06452732100537495, 'count': 3}
parent_3851fd4c3ffa {'title': '❚❚ Nikon User Support', 'max_score': 0.0625, 'mean_score': 0.05107449602095936, 'count': 3}
parent_64db013fd03f {'title': 'Nikon Z 5 Digital Camera', 'max_score': 0.05819659466641946, 'mean_score': 0.03673073945433376, 'count': 4}
parent_b441e0037e11 {'title': 'NIKKOR Z 24-50mm f/4-6.3 Lens', 'max_score': 0.03007688828584351, 'mean_score': 0.03007688828584351, 'count': 1}
parent_8c2ef09baa9f {'title': 'Parts of the Lens: Names and Functions', 'max_score': 0.029411764705882353, 'mean_score': 0.02430285547054643, 'count': 3}
parent_b5a7358f444b {'title': 'NIKKOR Z 24-200mm f/4-6.3 VR Lens', 'max_score': 0.015151515151515152, 'mean_score': 0.015151515151515152, 'count': 1}


In [81]:
# ============================================================
# 7. TOP-K PARENTS + EXPAND parent -> children
# ============================================================
ranked_parents = sorted(
    parent_scores.items(),
    key=lambda kv: (kv[1]["max_score"], kv[1]["mean_score"], kv[1]["count"]),
    reverse=True,
)[:TOP_PARENTS]

print(f"TOP {TOP_PARENTS} parents:")
for pid, info in ranked_parents:
    print(f"- {pid} | {info['title']} | max={info['max_score']:.4f} "
          f"mean={info['mean_score']:.4f} count={info['count']}")

expanded_docs = []
for pid, _info in ranked_parents:
    children = sorted(parent_groups[pid], key=lambda it: it["doc"].metadata.get("chunk_index", 0))
    if len(children) > MAX_CHILD_PER_PARENT:
        # uu tien children co fused score cao nhat khi parent qua to
        children = sorted(children, key=lambda it: it["score"], reverse=True)[:MAX_CHILD_PER_PARENT]
        children = sorted(children, key=lambda it: it["doc"].metadata.get("chunk_index", 0))
    for it in children:
        expanded_docs.append(it["doc"])
    print(f"expanded {pid}: {len(children)} children")

print(f"\ntotal expanded docs for rerank: {len(expanded_docs)}")


TOP 3 parents:
- parent_2bdaae1b71e4 | Specifications | max=0.0651 mean=0.0645 count=3
- parent_3851fd4c3ffa | ❚❚ Nikon User Support | max=0.0625 mean=0.0511 count=3
- parent_64db013fd03f | Nikon Z 5 Digital Camera | max=0.0582 mean=0.0367 count=4
expanded parent_2bdaae1b71e4: 3 children
expanded parent_3851fd4c3ffa: 3 children
expanded parent_64db013fd03f: 4 children

total expanded docs for rerank: 10


In [82]:
from sentence_transformers import CrossEncoder

# Load reranker
reranker = CrossEncoder(
    RERANKER_MODEL,  # lay tu cell Config
)

pairs = [
    (query_norm, doc.page_content)  # rerank bang query GOC da chuan hoa (chong drift tu paraphrase)
    for doc in expanded_docs
]

rerank_scores = reranker.predict(pairs)

reranked = sorted(
    zip(expanded_docs, rerank_scores),
    key=lambda x: x[1],
    reverse=True
)

for rank, (doc, score) in enumerate(reranked[:8], 1):
    print(
        rank,
        f"{score:.4f}",
        doc.metadata["chunk_id"],
        doc.metadata["parent_title"],
        doc.metadata.get("page_nos")
    )


Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

1 6.2500 chunk_0114 Specifications [70]
2 5.8750 chunk_0012 ❚❚ Nikon User Support [7]
3 5.6250 chunk_0120 Specifications [75]
4 5.3125 chunk_0117 Specifications [72, 73]
5 -2.6250 chunk_0070 Nikon Z 5 Digital Camera [42, 43]
6 -4.3125 chunk_0011 ❚❚ Nikon User Support [7]
7 -4.6875 chunk_0074 Nikon Z 5 Digital Camera [47]
8 -5.0000 chunk_0013 ❚❚ Nikon User Support [7, 8]


In [83]:
top_docs = [
    doc
    for doc, score in reranked[:TOP_N]  # TOP_N context cuoi dua vao LLM
]


In [84]:
from collections import Counter

parent_counts = Counter(
    doc.metadata["parent_id"]
    for doc in top_docs
)

print(parent_counts)

Counter({'parent_2bdaae1b71e4': 3, 'parent_3851fd4c3ffa': 1, 'parent_64db013fd03f': 1})


In [85]:
from collections import defaultdict

grouped = defaultdict(list)

for doc, score in reranked[:TOP_N]:
    grouped[doc.metadata["parent_id"]].append(
        (doc, score)
    )

for parent_id, items in grouped.items():
    print("\n", "=" * 80)
    print("Parent:", items[0][0].metadata["parent_title"])

    for doc, score in items:
        print(
            f"{doc.metadata['chunk_id']} "
            f"| score={score:.4f} "
            f"| pages={doc.metadata.get('page_nos')}"
        )


Parent: Specifications
chunk_0114 | score=6.2500 | pages=[70]
chunk_0120 | score=5.6250 | pages=[75]
chunk_0117 | score=5.3125 | pages=[72, 73]

Parent: ❚❚ Nikon User Support
chunk_0012 | score=5.8750 | pages=[7]

Parent: Nikon Z 5 Digital Camera
chunk_0070 | score=-2.6250 | pages=[42, 43]


In [86]:
parent_scores = {}

for parent_id, items in grouped.items():
    scores = [score for _, score in items]

    parent_scores[parent_id] = {
        "max_score": max(scores),
        "mean_score": sum(scores) / len(scores),
        "count": len(scores),
    }

for parent_id, info in parent_scores.items():
    print(parent_id, info)

parent_2bdaae1b71e4 {'max_score': 6.25, 'mean_score': 5.729166666666667, 'count': 3}
parent_3851fd4c3ffa {'max_score': 5.875, 'mean_score': 5.875, 'count': 1}
parent_64db013fd03f {'max_score': -2.625, 'mean_score': -2.625, 'count': 1}


In [87]:
# ============================================================
# 8. TOP-N CONTEXT — build context cho LLM
# ============================================================
context_parts = []
for i, doc in enumerate(top_docs, start=1):
    m = doc.metadata
    header = (
        f"[CTX {i} | {m.get('chunk_id')} | pages={m.get('page_nos')} "
        f"| parent={m.get('parent_title')}]"
    )
    context_parts.append(f"{header}\n{doc.page_content}")

context = "\n\n---\n\n".join(context_parts)
print(context[:2000])
print(f"\n... ({len(context)} chars, {len(top_docs)} docs)")


[CTX 1 | chunk_0114 | pages=[70] | parent=Specifications]
Specifications
Focal length, Nikon Z mount = 24-70mm. Maximumaperture, Nikon Z mount = f/4. Lens construction, Nikon Z mount = 14 elements in 11 groups (including 1 ED lens element, 1 aspherical ED element, 3 aspherical elements, elements with Nano Crystal Coat, and a fluorine-coated front lens element). Angle of view, Nikon Z mount = • FX format: 84° - 34° 20´ • DX format: 61° - 22° 50´. Focal length scale, Nikon Z mount = Graduated in millimeters (24, 28, 35, 50, 70). Focusing system, Nikon Z mount = Internal focusing system. Minimumfocus distance, Nikon Z mount = 0.3 m(0.99 ft) from focal plane at all zoom positions. Diaphragm blades, Nikon Z mount = 7 (rounded diaphragm opening). Aperture range, Nikon Z mount = f/4 - 22. Filter-attachment size, Nikon Z mount = 72 mm(P=0.75 mm). Dimensions, Nikon Z mount = Approx. 77.5 mmmaximumdiameter × 88.5mm (distance from camera lens mount flange when lens is retracted). Weight, Nikon Z 

In [90]:
# ============================================================
# 9. LLM ANSWER (English)
# ============================================================
system_prompt = (
    "You are a helpful assistant answering questions about a product manual. "
    "Answer IN ENGLISH using ONLY the provided context. "
    "Cite sources as [chunk_id] and page numbers when relevant. "
    "If the context does not contain the answer, say so explicitly."
)

response = llm.invoke([
    ("system", system_prompt),
    ("human", f"Context:\n{context}\n\nQuestion: {query_norm}"),
])
answer=response.content

print(answer)


The Nikon Z 24-70 f/4 and Nikon Z 24-200 f/4 -f/6.3 differ in several key aspects:  

1. **Focal Range**:  
   - The 24-70 f/4 offers a narrower range (24–70mm) compared to the 24-200 f/4 -f/6.3 (24–200mm), providing greater versatility for different shooting scenarios.  

2. **Aperture Range**:  
   - The 24-70 f/4 has a fixed maximum aperture of **f/4**, while the 24-200 f/4 -f/6.3 has a wider range (f/4–f/6.3), allowing for more flexibility in low-light conditions.  

3. **Optical Quality**:  
   - The 24-200 f/4 -f/6.3 features **19 elements in 15 groups** (2 ED, 1 aspherical ED, 2 aspherical, and ARNEO coat), while the 24-70 has **14 elements in 11 groups** (1 ED, 1 aspherical ED, 3 aspherical, Nano Crystal Coat, and fluorine coating). The 24-200 likely offers superior optical performance due to more elements and groups.  

4. **Dimensions and Weight**:  
   - The 24-200 f/4 -f/6.3 has a **larger maximum diameter (76.5 mm)** and a **heavier weight (570 g)** compared to the 24-70 f

In [93]:
print(response)

content="The Nikon Z 24-70 f/4 and Nikon Z 24-200 f/4 -f/6.3 differ in several key aspects:  \n\n1. **Focal Range**:  \n   - The 24-70 f/4 offers a narrower range (24–70mm) compared to the 24-200 f/4 -f/6.3 (24–200mm), providing greater versatility for different shooting scenarios.  \n\n2. **Aperture Range**:  \n   - The 24-70 f/4 has a fixed maximum aperture of **f/4**, while the 24-200 f/4 -f/6.3 has a wider range (f/4–f/6.3), allowing for more flexibility in low-light conditions.  \n\n3. **Optical Quality**:  \n   - The 24-200 f/4 -f/6.3 features **19 elements in 15 groups** (2 ED, 1 aspherical ED, 2 aspherical, and ARNEO coat), while the 24-70 has **14 elements in 11 groups** (1 ED, 1 aspherical ED, 3 aspherical, Nano Crystal Coat, and fluorine coating). The 24-200 likely offers superior optical performance due to more elements and groups.  \n\n4. **Dimensions and Weight**:  \n   - The 24-200 f/4 -f/6.3 has a **larger maximum diameter (76.5 mm)** and a **heavier weight (570 g)** co